# VisualNews Metadata Inspection

Inspect local metadata in memory. Raw files stay unchanged; no mapping or asset staging.

## Setup

In [1]:
import json
from pathlib import Path
from collections import Counter

import pandas as pd

cwd = Path.cwd()
root = cwd.parent if cwd.name == "notebooks" else cwd
raw_root = root / "data" / "raw" / "visualnews_metadata"
train_path = raw_root / "train.json"
val_path = raw_root / "val.json"
test_path = raw_root / "test.json"
paths = {"train": train_path, "val": val_path, "test": test_path}

In [2]:
file_rows = []
for split, path in paths.items():
    exists = path.is_file()
    readable = False
    if exists:
        with path.open("r", encoding="utf-8") as handle:
            handle.read(1)
        readable = True
    file_rows.append({
        "split": split, "path": str(path.relative_to(root)), "exists": exists,
        "size_mb": round(path.stat().st_size / 1_000_000, 3) if exists else None,
        "readable": readable,
    })
display(pd.DataFrame(file_rows))

,split,path,exists,size_mb,readable
0,train,data/raw/visualnews_metadata/train.json,True,351.984,True
1,val,data/raw/visualnews_metadata/val.json,True,37.288,True
2,test,data/raw/visualnews_metadata/test.json,True,37.370,True


## Load Data

In [3]:
objects = {}
for split, path in paths.items():
    with path.open("r", encoding="utf-8") as handle:
        objects[split] = json.load(handle)

## Structure

In [4]:
structure_rows = []
for split, obj in objects.items():
    structure_rows.append({
        "split": split, "top_level_type": type(obj).__name__,
        "top_level_items": len(obj) if isinstance(obj, (dict, list)) else None,
        "first_keys": list(obj)[:3] if isinstance(obj, dict) else None,
        "value_types": dict(Counter(type(value).__name__ for value in obj.values())) if isinstance(obj, dict) else None,
        "numeric_keys": all(type(key) is str and key.isdecimal() for key in obj) if isinstance(obj, dict) else None,
    })
structure_table = pd.DataFrame(structure_rows)
display(structure_table)

,split,top_level_type,top_level_items,first_keys,value_types,numeric_keys
0,train,dict,385003,"[728421, 830076, 592744]",{'dict': 385003},True
1,val,dict,40830,"[931806, 389177, 226877]",{'dict': 40830},True
2,test,dict,40915,"[701864, 759902, 893678]",{'dict': 40915},True


The local files are ID-keyed dictionaries. Use their values as inspection records.

In [5]:
records = {}
malformed_counts = {}
key_mismatches = {}
for split, obj in objects.items():
    if not isinstance(obj, dict) or not all(type(key) is str and key.isdecimal() for key in obj):
        raise TypeError(f"{split}: structure changed; review before normalizing")
    malformed_counts[split] = sum(not isinstance(value, dict) for value in obj.values())
    records[split] = [value for value in obj.values() if isinstance(value, dict)]
    key_mismatches[split] = sum(str(value.get("id")) != key for key, value in obj.items() if isinstance(value, dict))

normalization_table = pd.DataFrame([
    {"split": split, "raw_items": len(objects[split]), "inspection_records": len(rows),
     "malformed_records": malformed_counts[split], "key_id_mismatches": key_mismatches[split]}
    for split, rows in records.items()
])
display(normalization_table)

,split,raw_items,inspection_records,malformed_records,key_id_mismatches
0,train,385003,385003,0,0
1,val,40830,40830,0,0
2,test,40915,40915,0,0


## Sample Records

In [6]:
sample_columns = ["id", "caption", "image_path", "article_path", "source", "timestamp", "topic", "title"]
for split, rows in records.items():
    print(split)
    sample = pd.DataFrame(rows[:3]).reindex(columns=sample_columns)
    with pd.option_context("display.max_colwidth", 90):
        display(sample)

train


,id,caption,image_path,article_path,source,timestamp,topic,title
0,728421,Anthony Stokes last played a league game for Celtic in August,visual_news/origin/guardian/images/0540/632.jpg,visual_news/origin/guardian/articles/728421.txt,guardian,2015-12-02T11:33:37Z,football,Anthony Stokes suspended by Celtic for two weeks after Twitter rant
1,830076,Kell Brook successfully defended his title against Frankie Gavin in May but Diego Chav...,visual_news/origin/guardian/images/0435/014.jpg,visual_news/origin/guardian/articles/830076.txt,guardian,2015-09-02T16:35:10Z,sport,Kell Brook to defend world title against Diego Chaves in Sheffield
2,592744,Mohamed ElBaradei has been widely tipped to take up the post in the wake of Morsi s ou...,visual_news/origin/guardian/images/0318/914.jpg,visual_news/origin/guardian/articles/592744.txt,guardian,2013-07-06T23:44:00Z,world,Mohamed ElBaradei's appointment as Egypt's interim PM thrown into doubt


val


,id,caption,image_path,article_path,source,timestamp,topic,title
0,931806,People might assume that I m rolling in money Couldn t be further from the truth at th...,visual_news/origin/guardian/images/0148/129.jpg,visual_news/origin/guardian/articles/931806.txt,guardian,2014-10-06T08:00:09Z,film,"Will Poulter: ‘Hanging out in Soho House LA, that’s my worst nightmare’"
1,389177,Garth Brooks performs during the Opening Ceremonies concert at the Lincoln Memorial fo...,visual_news/origin/usa_today/images/0707/622.jpg,visual_news/origin/usa_today/articles/389177.txt,usa_today,2016-10-19T12:03:03Z,science_technology,"You can finally stream Garth Brooks' music, but only on Amazon"
2,226877,Game 5 in Los Angeles Dodgers shortstop Corey Seager singles in the first inning,visual_news/origin/usa_today/images/0706/069.jpg,visual_news/origin/usa_today/articles/226877.txt,usa_today,2016-10-20T04:19:04Z,sports,How Anthony Rizzo broke out of his postseason slump


test


,id,caption,image_path,article_path,source,timestamp,topic,title
0,701864,Jim ONeill Goldman Sachs chief economist is part of a group seeking to wrest control o...,visual_news/origin/guardian/images/0515/179.jpg,visual_news/origin/guardian/articles/701864.txt,guardian,2010-03-04T20:55:00Z,football,Jim O'Neill faces red card from Goldman Sachs
1,759902,Gary Neville was pictured carrying notes that appeared to reveal England players instr...,visual_news/origin/guardian/images/0567/321.jpg,visual_news/origin/guardian/articles/759902.txt,guardian,2014-06-11T23:18:23Z,football,Gary Neville gaffe reveals England ‘strategy’ for Italy game
2,893678,Mark Cavendish and Bradley Wiggins handsling as they break away and go on to win the M...,visual_news/origin/guardian/images/0119/773.jpg,visual_news/origin/guardian/articles/893678.txt,guardian,2016-03-07T16:25:41Z,sport,Mark Cavendish conundrum now the toughest problem for GB cycling


## Record Counts

These files are the NewsCLIPpings-author VisualNews-derived metadata subset, not the complete VisualNews corpus.

In [7]:
def valid_id(value):
    return type(value) is int and value >= 0


def has_text(value):
    return type(value) is str and bool(value.strip())


def field_counts(rows, field):
    values = (row.get(field) for row in rows)
    valid = valid_id if field == "id" else has_text
    return Counter(value for value in values if valid(value))

count_table = pd.DataFrame([
    {"split": split, "record_count": len(rows), "unique_id_count": len(field_counts(rows, "id"))}
    for split, rows in records.items()
])
display(count_table)
print("Total metadata records:", f"{sum(len(rows) for rows in records.values()):,}")

,split,record_count,unique_id_count
0,train,385003,385003
1,val,40830,40830
2,test,40915,40915


Total metadata records: 466,748


## Fields

In [8]:
split_fields = {split: set(field for row in rows for field in row) for split, rows in records.items()}
fields = list(dict.fromkeys(field for rows in records.values() for row in rows for field in row))
field_table = pd.DataFrame([
    {"field": field, **{f"present_in_{split}": field in names for split, names in split_fields.items()}}
    for field in fields
])
display(field_table)
print("Split field sets identical:", split_fields["train"] == split_fields["val"] == split_fields["test"])

,field,present_in_train,present_in_val,present_in_test
0,id,True,True,True
1,caption,True,True,True
2,image_path,True,True,True
3,article_path,True,True,True
4,full_article_path,True,True,True
5,caption_entities_spacy,True,True,True
6,caption_entities_rel,True,True,True
7,image_has_person,True,True,True
8,topic,True,True,True
9,source,True,True,True


Split field sets identical: True


In [9]:
type_rows = []
for split, rows in records.items():
    for field in fields:
        types = Counter(type(row[field]).__name__ for row in rows if field in row)
        type_rows.append({"split": split, "field": field, "types_seen": dict(types)})
type_table = pd.DataFrame(type_rows)
display(type_table.pivot(index="field", columns="split", values="types_seen").reindex(fields))

split,test,train,val
field,,,
id,{'int': 40915},{'int': 385003},{'int': 40830}
caption,{'str': 40915},{'str': 385003},{'str': 40830}
image_path,{'str': 40915},{'str': 385003},{'str': 40830}
article_path,{'str': 40915},{'str': 385003},{'str': 40830}
full_article_path,{'str': 40915},{'str': 385003},{'str': 40830}
caption_entities_spacy,{'list': 40915},{'list': 385003},{'list': 40830}
caption_entities_rel,{'list': 40915},{'list': 385003},{'list': 40830}
image_has_person,{'bool': 40915},{'bool': 385003},{'bool': 40830}
topic,{'str': 40915},{'str': 385003},{'str': 40830}


## Missing Values

In [10]:
missing_rows = []
for split, rows in records.items():
    for field in fields:
        values = [row[field] for row in rows if field in row]
        missing_rows.append({
            "split": split, "field": field, "missing_key_count": len(rows) - len(values),
            "null_count": sum(value is None for value in values),
            "empty_string_count": sum(type(value) is str and value == "" for value in values),
            "whitespace_only_count": sum(type(value) is str and value != "" and not value.strip() for value in values),
        })
missing_table = pd.DataFrame(missing_rows)
display(missing_table)

,split,field,missing_key_count,null_count,empty_string_count,whitespace_only_count
0,train,id,0,0,0,0
1,train,caption,0,0,0,0
2,train,image_path,0,0,0,0
3,train,article_path,0,0,0,0
4,train,full_article_path,0,0,0,0
5,train,caption_entities_spacy,0,0,0,0
6,train,caption_entities_rel,0,0,0,0
7,train,image_has_person,0,0,0,0
8,train,topic,0,0,0,0
9,train,source,0,0,0,0


## IDs

ID uniqueness alone does not establish image uniqueness.

In [11]:
id_rows = []
for split, rows in records.items():
    counts = field_counts(rows, "id")
    id_rows.append({
        "split": split, "unique_ids": len(counts),
        "duplicate_id_rows": sum(count - 1 for count in counts.values()),
        "id_types": dict(Counter(type(row.get("id")).__name__ for row in rows)),
        "malformed_or_missing_ids": sum(not valid_id(row.get("id")) for row in rows),
        "key_id_mismatches": key_mismatches[split],
    })
id_table = pd.DataFrame(id_rows)
display(id_table)

pairs = [("train", "val"), ("train", "test"), ("val", "test")]
id_sets = {split: set(field_counts(rows, "id")) for split, rows in records.items()}
id_overlap_table = pd.DataFrame([
    {"split_pair": f"{left} ∩ {right}", "shared_ids": len(id_sets[left] & id_sets[right])}
    for left, right in pairs
])
display(id_overlap_table)

,split,unique_ids,duplicate_id_rows,id_types,malformed_or_missing_ids,key_id_mismatches
0,train,385003,0,{'int': 385003},0,0
1,val,40830,0,{'int': 40830},0,0
2,test,40915,0,{'int': 40915},0,0


,split_pair,shared_ids
0,train ∩ val,0
1,train ∩ test,0
2,val ∩ test,0


## Image Paths

Metadata path present does not mean the image file is locally available.

In [12]:
def malformed_path(value):
    if not has_text(value):
        return False
    path = Path(value)
    return path.is_absolute() or ".." in path.parts or "\\" in value or "\x00" in value or not path.suffix


def path_stats(split, rows, field):
    counts = field_counts(rows, field)
    values = [row.get(field) for row in rows]
    return {
        "split": split, "field": field,
        "missing_or_null": sum(field not in row or row[field] is None for row in rows),
        "empty_or_whitespace": sum(type(value) is str and not value.strip() for value in values),
        "unique_paths": len(counts), "duplicate_path_rows": sum(count - 1 for count in counts.values()),
        "malformed_paths": sum(malformed_path(value) for value in values),
        "extensions": dict(Counter(Path(value).suffix.lower() for value in values if has_text(value))),
    }

image_path_table = pd.DataFrame([path_stats(split, rows, "image_path") for split, rows in records.items()])
display(image_path_table)

,split,field,missing_or_null,empty_or_whitespace,unique_paths,duplicate_path_rows,malformed_paths,extensions
0,train,image_path,0,0,385003,0,0,{'.jpg': 385003}
1,val,image_path,0,0,40830,0,0,{'.jpg': 40830}
2,test,image_path,0,0,40915,0,0,{'.jpg': 40915}


In [13]:
pattern_rows = []
image_examples = []
for split, rows in records.items():
    patterns = Counter()
    for row in rows:
        value = row.get("image_path")
        if has_text(value):
            prefix = "/".join(Path(value).parts[:4])
            patterns[(row.get("source"), prefix)] += 1
    pattern_rows.extend({"split": split, "source": source, "path_prefix": prefix, "count": count} for (source, prefix), count in patterns.items())
    image_examples.extend({"split": split, "image_path": value} for value in list(field_counts(rows, "image_path"))[:2])
display(pd.DataFrame(pattern_rows))
display(pd.DataFrame(image_examples))

,split,source,path_prefix,count
0,train,guardian,visual_news/origin/guardian/images,185203
1,train,usa_today,visual_news/origin/usa_today/images,81249
2,train,bbc,visual_news/origin/bbc/images,59270
3,train,washington_post,visual_news/origin/washington_post/images,59281
4,val,guardian,visual_news/origin/guardian/images,19642
5,val,usa_today,visual_news/origin/usa_today/images,8607
6,val,washington_post,visual_news/origin/washington_post/images,6234
7,val,bbc,visual_news/origin/bbc/images,6347
8,test,guardian,visual_news/origin/guardian/images,19596
9,test,usa_today,visual_news/origin/usa_today/images,8703


,split,image_path
0,train,visual_news/origin/guardian/images/0540/632.jpg
1,train,visual_news/origin/guardian/images/0435/014.jpg
2,val,visual_news/origin/guardian/images/0148/129.jpg
3,val,visual_news/origin/usa_today/images/0707/622.jpg
4,test,visual_news/origin/guardian/images/0515/179.jpg
5,test,visual_news/origin/guardian/images/0567/321.jpg


## Article Paths

Inspect both path schemes separately; matching records do not establish asset equivalence. No physical files are checked.

In [14]:
article_path_table = pd.DataFrame([
    path_stats(split, rows, field)
    for split, rows in records.items() for field in ("article_path", "full_article_path")
])
display(article_path_table)

article_examples = []
for split, rows in records.items():
    for field in ("article_path", "full_article_path"):
        for value in list(field_counts(rows, field))[:2]:
            article_examples.append({"split": split, "field": field, "path_example": value})
with pd.option_context("display.max_colwidth", 180):
    display(pd.DataFrame(article_examples))

,split,field,missing_or_null,empty_or_whitespace,unique_paths,duplicate_path_rows,malformed_paths,extensions
0,train,article_path,0,0,385003,0,0,{'.txt': 385003}
1,train,full_article_path,0,0,237309,147694,0,{'.json': 385003}
2,val,article_path,0,0,40830,0,0,{'.txt': 40830}
3,val,full_article_path,0,0,34469,6361,0,{'.json': 40830}
4,test,article_path,0,0,40915,0,0,{'.txt': 40915}
5,test,full_article_path,0,0,34470,6445,0,{'.json': 40915}


,split,field,path_example
0,train,article_path,visual_news/origin/guardian/articles/728421.txt
1,train,article_path,visual_news/origin/guardian/articles/830076.txt
2,train,full_article_path,visual_news/articles/guardian/http:::mobile-apps.guardianapis.com:items:football:2015:dec:02:anthony-stokes-suspended-celtic-twitter-inverness-200.json
3,train,full_article_path,visual_news/articles/guardian/http:::mobile-apps.guardianapis.com:items:sport:2015:sep:02:kell-brook-world-title-diego-chaves-sheffield-200.json
4,val,article_path,visual_news/origin/guardian/articles/931806.txt
5,val,article_path,visual_news/origin/usa_today/articles/389177.txt
6,val,full_article_path,visual_news/articles/guardian/http:::mobile-apps.guardianapis.com:items:film:2014:oct:06:will-poulter-interview-maze-runner-200.json
7,val,full_article_path,visual_news/articles/usa_today/92400414.json
8,test,article_path,visual_news/origin/guardian/articles/701864.txt
9,test,article_path,visual_news/origin/guardian/articles/759902.txt


## Captions

Lengths are character counts. Duplicate counts exclude missing/blank values and count occurrences beyond the first.

In [15]:
caption_rows = []
for split, rows in records.items():
    values = [row.get("caption") for row in rows]
    lengths = pd.Series([len(value) for value in values if has_text(value)], dtype="int64")
    counts = field_counts(rows, "caption")
    caption_rows.append({
        "split": split, "missing_or_null": sum(value is None for value in values),
        "empty_or_whitespace": sum(type(value) is str and not value.strip() for value in values),
        "exact_duplicate_rows": sum(count - 1 for count in counts.values()),
        "count": len(lengths), "min_chars": lengths.min(), "max_chars": lengths.max(),
        "mean_chars": lengths.mean(), "median_chars": lengths.median(),
    })
caption_table = pd.DataFrame(caption_rows)
display(caption_table)

,split,missing_or_null,empty_or_whitespace,exact_duplicate_rows,count,min_chars,max_chars,mean_chars,median_chars
0,train,0,0,1115,385003,18,292,99.028745,96.0
1,val,0,0,18,40830,19,350,98.431056,96.0
2,test,0,0,13,40915,20,451,98.876720,96.0


## Sources

In [16]:
source_counts = {split: field_counts(rows, "source") for split, rows in records.items()}
source_table = pd.DataFrame(source_counts).fillna(0).astype(int)
source_table["total"] = source_table.sum(axis=1)
source_table.index.name = "source"
source_table = source_table.sort_values("total", ascending=False)
display(source_table)
print("Observed source values:", source_table.index.tolist())

,train,val,test,total
source,,,,
guardian,185203,19642,19596,224441
usa_today,81249,8607,8703,98559
bbc,59270,6347,6371,71988
washington_post,59281,6234,6245,71760


Observed source values: ['guardian', 'usa_today', 'bbc', 'washington_post']


## Timestamps

Article-associated timestamps. Timestamp should not later be treated automatically as event or capture date.

In [17]:
timestamp_rows = []
parsed_timestamps = {}
for split, rows in records.items():
    values = [row.get("timestamp") for row in rows]
    nonempty = {index: value for index, value in enumerate(values) if has_text(value)}
    parsed = pd.to_datetime(pd.Series(nonempty, dtype="object"), format="mixed", errors="coerce", utc=True)
    parsed_timestamps[split] = parsed
    timestamp_rows.append({
        "split": split, "types_seen": dict(Counter(type(value).__name__ for value in values)),
        "missing_or_null": sum(value is None for value in values),
        "empty_or_whitespace": sum(type(value) is str and not value.strip() for value in values),
        "invalid_nonstring_values": sum(value is not None and type(value) is not str for value in values),
        "parseable": int(parsed.notna().sum()), "unparseable_nonempty": int(parsed.isna().sum()),
        "min_timestamp": parsed.min(), "max_timestamp": parsed.max(),
    })
timestamp_table = pd.DataFrame(timestamp_rows)
display(timestamp_table)

,split,types_seen,missing_or_null,empty_or_whitespace,invalid_nonstring_values,parseable,unparseable_nonempty,min_timestamp,max_timestamp
0,train,{'str': 385003},0,0,0,385003,0,1963-08-29 17:14:00+00:00,2016-11-07 22:48:48+00:00
1,val,{'str': 40830},0,0,0,40830,0,1900-01-01 00:00:00+00:00,2016-11-07 21:33:00+00:00
2,test,{'str': 40915},0,0,0,40915,0,1963-08-29 17:14:00+00:00,2016-11-07 11:00:01+00:00


In [18]:
oldest_rows = []
for split, parsed in parsed_timestamps.items():
    for index, timestamp in parsed.nsmallest(3).items():
        row = records[split][index]
        oldest_rows.append({
            "split": split, "id": row.get("id"), "timestamp": timestamp,
            "source": row.get("source"), "title": row.get("title"),
        })
with pd.option_context("display.max_colwidth", 100):
    display(pd.DataFrame(oldest_rows))

,split,id,timestamp,source,title
0,train,143712,1963-08-29 17:14:00+00:00,washington_post,"200,000 Jam Mall in Mammoth Rally In Solemn, Orderly Plea for Equality"
1,train,54401,1963-08-29 17:14:00+00:00,washington_post,"200,000 Jam Mall in Mammoth Rally In Solemn, Orderly Plea for Equality"
2,train,165789,1963-08-29 17:14:00+00:00,washington_post,"200,000 Jam Mall in Mammoth Rally In Solemn, Orderly Plea for Equality"
3,val,524082,1900-01-01 00:00:00+00:00,guardian,Japan's factory output and consumer spending plummet
4,val,137767,1963-08-29 17:14:00+00:00,washington_post,"200,000 Jam Mall in Mammoth Rally In Solemn, Orderly Plea for Equality"
5,val,794914,2007-08-09 16:49:14+00:00,guardian,Who is the real Fake Steve Jobs?
6,test,49251,1963-08-29 17:14:00+00:00,washington_post,"200,000 Jam Mall in Mammoth Rally In Solemn, Orderly Plea for Equality"
7,test,51922,1963-08-29 17:14:00+00:00,washington_post,"200,000 Jam Mall in Mammoth Rally In Solemn, Orderly Plea for Equality"
8,test,92161,1963-08-29 17:14:00+00:00,washington_post,"200,000 Jam Mall in Mammoth Rally In Solemn, Orderly Plea for Equality"


## Topics

In [19]:
topic_counts = {split: field_counts(rows, "topic") for split, rows in records.items()}
topic_table = pd.DataFrame(topic_counts).fillna(0).astype(int)
topic_table["total"] = topic_table.sum(axis=1)
topic_table.index.name = "topic"
topic_table = topic_table.sort_values("total", ascending=False)
with pd.option_context("display.max_rows", None):
    display(topic_table)

topic_summary = pd.DataFrame([
    {"split": split, "unique_topics": len(counts),
     "missing_or_null": sum(row.get("topic") is None for row in records[split]),
     "empty_or_whitespace": sum(type(row.get("topic")) is str and not row["topic"].strip() for row in records[split])}
    for split, counts in topic_counts.items()
])
display(topic_summary)

,train,val,test,total
topic,,,,
arts_culture,72898,7686,7793,88377
world,28716,3033,2898,34647
international_relations,27314,2887,2941,33142
law_crime,27166,2786,2901,32853
science_technology,20386,2159,2134,24679
football,20308,2140,2167,24615
sport,18820,1972,2048,22840
politics_elections,14619,1566,1551,17736
sports,14496,1576,1521,17593


,split,unique_topics,missing_or_null,empty_or_whitespace
0,train,94,0,0
1,val,83,0,0
2,test,79,0,0


## Titles

In [20]:
title_rows = []
for split, rows in records.items():
    values = [row.get("title") for row in rows]
    counts = field_counts(rows, "title")
    missing = sum(value is None for value in values)
    title_rows.append({
        "split": split, "missing_or_null": missing,
        "missing_percentage": round(100 * missing / len(rows), 4) if rows else None,
        "empty_or_whitespace": sum(type(value) is str and not value.strip() for value in values),
        "exact_duplicate_rows": sum(count - 1 for count in counts.values()),
    })
title_table = pd.DataFrame(title_rows)
display(title_table)
display(pd.DataFrame([
    {"split": split, "title": title}
    for split, rows in records.items() for title in list(field_counts(rows, "title"))[:2]
]))

,split,missing_or_null,missing_percentage,empty_or_whitespace,exact_duplicate_rows
0,train,112,0.0291,0,150092
1,val,7,0.0171,0,6888
2,test,8,0.0196,0,6983


,split,title
0,train,Anthony Stokes suspended by Celtic for two wee...
1,train,Kell Brook to defend world title against Diego...
2,val,"Will Poulter: ‘Hanging out in Soho House LA, t..."
3,val,"You can finally stream Garth Brooks' music, bu..."
4,test,Jim O'Neill faces red card from Goldman Sachs
5,test,Gary Neville gaffe reveals England ‘strategy’ ...


## Entities

Inspect structure only; no entity normalization or modeling.

In [21]:
entity_fields = ["caption_entities_spacy", "caption_entities_rel", "title_entities_spacy"]
entity_rows = []
for split, rows in records.items():
    for field in entity_fields:
        values = [row.get(field) for row in rows]
        lists = [value for value in values if type(value) is list]
        entry_types = Counter(type(entry).__name__ for value in lists for entry in value)
        entry_lengths = Counter(len(entry) for value in lists for entry in value if isinstance(entry, (list, dict)))
        example = next((value[:2] for value in lists if value), [])
        entity_rows.append({
            "split": split, "field": field, "field_types": dict(Counter(type(value).__name__ for value in values)),
            "missing_or_null": sum(value is None for value in values),
            "empty_lists": sum(not value for value in lists),
            "min_entries": min(map(len, lists), default=None), "max_entries": max(map(len, lists), default=None),
            "entry_types": dict(entry_types), "entry_lengths": dict(entry_lengths),
            "example_first_two": json.dumps(example, ensure_ascii=False)[:240],
        })
entity_table = pd.DataFrame(entity_rows)
with pd.option_context("display.max_colwidth", 100):
    display(entity_table)

,split,field,field_types,missing_or_null,empty_lists,min_entries,max_entries,entry_types,entry_lengths,example_first_two
0,train,caption_entities_spacy,{'list': 385003},0,0,2,11,{'list': 1079918},{2: 1079918},"[[""Celtic"", ""NORP""], [""Anthony Stokes"", ""PERSON""]]"
1,train,caption_entities_rel,{'list': 385003},0,16108,0,10,{'list': 910931},{4: 910931},"[[""Anthony_Stokes"", ""PER"", ""Anthony Stokes"", 0.9918], [""Celtic_F.C."", ""ORG"", ""Celtic"", 0.9935]]"
2,train,title_entities_spacy,{'list': 385003},0,62256,0,8,{'list': 557917},{2: 557917},"[[""Twitter"", ""ORG""], [""Anthony Stokes"", ""PERSON""]]"
3,val,caption_entities_spacy,{'list': 40830},0,0,2,9,{'list': 114367},{2: 114367},"[[""Will Poulter"", ""PERSON""], [""London"", ""GPE""]]"
4,val,caption_entities_rel,{'list': 40830},0,1714,0,10,{'list': 96521},{4: 96521},"[[""Will_Poulter"", ""PER"", ""Will Poulter"", 0.9183], [""London"", ""LOC"", ""London"", 0.9982]]"
5,val,title_entities_spacy,{'list': 40830},0,6614,0,8,{'list': 59250},{2: 59250},"[[""Will Poulter"", ""PERSON""], [""Soho House LA"", ""FAC""]]"
6,test,caption_entities_spacy,{'list': 40915},0,0,2,10,{'list': 114521},{2: 114521},"[[""Glazer"", ""PERSON""], [""Manchester United"", ""GPE""]]"
7,test,caption_entities_rel,{'list': 40915},0,1752,0,10,{'list': 96585},{4: 96585},"[[""Manchester_United_F.C."", ""ORG"", ""Manchester United"", 0.9549], [""Avram_Glazer"", ""ORG"", ""Glazer..."
8,test,title_entities_spacy,{'list': 40915},0,6519,0,6,{'list': 59403},{2: 59403},"[[""Jim O'Neill"", ""PERSON""], [""Goldman Sachs"", ""ORG""]]"


## Duplicates

Counts exclude invalid/missing/blank values; repeated rows are occurrences beyond the first. Nothing is removed.

In [22]:
duplicate_fields = ["id", "image_path", "caption", "article_path", "full_article_path", "title"]
duplicate_rows = []
for split, rows in records.items():
    for field in duplicate_fields:
        counts = field_counts(rows, field)
        duplicate_rows.append({
            "split": split, "field": field, "unique_values": len(counts),
            "duplicate_rows": sum(count - 1 for count in counts.values()),
            "duplicate_groups": sum(count > 1 for count in counts.values()),
        })
duplicate_table = pd.DataFrame(duplicate_rows)
display(duplicate_table)

,split,field,unique_values,duplicate_rows,duplicate_groups
0,train,id,385003,0,0
1,train,image_path,385003,0,0
2,train,caption,383888,1115,1044
3,train,article_path,385003,0,0
4,train,full_article_path,237309,147694,43449
5,train,title,234799,150092,42751
6,val,id,40830,0,0
7,val,image_path,40830,0,0
8,val,caption,40812,18,18
9,val,article_path,40830,0,0


## Split Overlap

Exact metadata overlap only; pixel duplication and event-level leakage remain unchecked.

In [23]:
overlap_rows = []
for field in ("id", "image_path", "caption", "article_path", "full_article_path"):
    sets = {split: set(field_counts(rows, field)) for split, rows in records.items()}
    for left, right in pairs:
        overlap_rows.append({
            "field": field, "split_pair": f"{left} ∩ {right}",
            "shared_unique_values": len(sets[left] & sets[right]),
        })
overlap_table = pd.DataFrame(overlap_rows)
display(overlap_table.pivot(index="field", columns="split_pair", values="shared_unique_values"))

split_pair,train ∩ test,train ∩ val,val ∩ test
field,,,
article_path,0,0,0
caption,0,0,0
full_article_path,13910,14104,5113
id,0,0,0
image_path,0,0,0


## Quality Checks

Check the observed schema and simple relative path rules. URL-like colons inside archive filenames are allowed.

In [24]:
allowed_types = {
    "id": (int,), "caption": (str,), "image_path": (str,), "article_path": (str,),
    "full_article_path": (str,), "caption_entities_spacy": (list,), "caption_entities_rel": (list,),
    "image_has_person": (bool,), "topic": (str,), "source": (str,), "timestamp": (str,),
    "title": (str, type(None)), "title_entities_spacy": (list,),
}
expected_sources = {"guardian", "bbc", "washington_post", "usa_today"}
quality_rows = []
for split, rows in records.items():
    id_stats = id_table[id_table["split"] == split].iloc[0]
    times = timestamp_table[timestamp_table["split"] == split].iloc[0]
    path_rows = [path_stats(split, rows, field) for field in ("image_path", "article_path", "full_article_path")]
    checks = {
        "malformed records": malformed_counts[split],
        "empty record dictionary": int(not objects[split]),
        "inconsistent record schemas": sum(set(row) != set(allowed_types) for row in rows),
        "invalid field types": sum(type(row[field]) not in types for row in rows for field, types in allowed_types.items() if field in row),
        "duplicate IDs beyond first": int(id_stats["duplicate_id_rows"]),
        "malformed or missing IDs": int(id_stats["malformed_or_missing_ids"]),
        "outer key / record ID mismatches": key_mismatches[split],
        "missing or blank captions": sum(not has_text(row.get("caption")) for row in rows),
        "missing or blank image paths": sum(not has_text(row.get("image_path")) for row in rows),
        "missing or blank article paths": sum(not has_text(row.get("article_path")) for row in rows),
        "missing or blank full article paths": sum(not has_text(row.get("full_article_path")) for row in rows),
        "unexpected source values": sum(type(row.get("source")) is not str or row["source"] not in expected_sources for row in rows),
        "invalid or missing timestamps": len(rows) - int(times["parseable"]),
        "malformed paths": sum(item["malformed_paths"] for item in path_rows),
        "missing or blank titles (optional)": sum(not has_text(row.get("title")) for row in rows),
    }
    quality_rows.extend({"split": split, "check": check, "count": count} for check, count in checks.items())
quality_table = pd.DataFrame(quality_rows)
display(quality_table.pivot(index="check", columns="split", values="count").reindex(columns=list(records)))

split,train,val,test
check,,,
duplicate IDs beyond first,0,0,0
empty record dictionary,0,0,0
inconsistent record schemas,0,0,0
invalid field types,0,0,0
invalid or missing timestamps,0,0,0
malformed or missing IDs,0,0,0
malformed paths,0,0,0
malformed records,0,0,0
missing or blank article paths,0,0,0


## Notes

- VisualNews-derived metadata inspected.
- No files were modified.
- Mapping has not been performed yet.
- Next step will be decided after reviewing the results.